# Train a Custom xG Model and Compare Spain-France Predictions

This notebook trains a local custom expected-goals model from raw StatsBomb shot events, then compares the Spain vs France pre-match prediction against the StatsBomb-provided xG benchmark from notebook 11.

The custom model is intentionally educational and transparent. It uses only pre-shot information such as shot location, angle, shot type, body part, technique, play pattern, and freeze-frame counts. It does **not** use leakage fields such as StatsBomb xG, shot outcome, shot end location, or post-shot save/post fields as model inputs.

Because `xgboost` is not installed in this environment, this notebook uses scikit-learn's `HistGradientBoostingClassifier`, a gradient-boosted tree model that is available locally.

## 1. Configuration

The training data uses the same senior men's historical proxy scope as the prior xG notebook: FIFA World Cup 2018 and 2022 plus UEFA Euro 2020 and 2024. Penalty shootouts are excluded by removing period 5 shots. The headline Spain-France comparison remains a 90-minute model using periods 1 and 2.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda text: text

TARGET_MATCH = {
    "team_a": "Spain",
    "team_b": "France",
    "match_name": "Spain vs France",
    "competition_name": "2026 FIFA World Cup",
    "round_name": "Semifinal",
    "match_date": "2026-07-14",
    "neutral_site": True,
}

DATA_DIR = Path("archive/data")
OUTPUT_DIR = Path("outputs/spain_france_custom_xg_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_COMPETITIONS = {
    ("FIFA World Cup", "2018"),
    ("FIFA World Cup", "2022"),
    ("UEFA Euro", "2020"),
    ("UEFA Euro", "2024"),
}

EXCLUDED_PERIODS = [5]
HEADLINE_PERIODS = [1, 2]
RECENT_MATCH_COUNT = 10
SHRINKAGE_MATCHES = 8
MAX_GOALS = 8
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print(f"StatsBomb data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(json.dumps(TARGET_MATCH, indent=2))

## 2. Load Selected StatsBomb Matches

This section reads local StatsBomb metadata and filters to the selected senior men's tournaments. No external downloads or current-tournament data are used.

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

if not DATA_DIR.exists():
    raise FileNotFoundError(f"StatsBomb data directory not found: {DATA_DIR}")

competitions = pd.DataFrame(load_json(DATA_DIR / "competitions.json"))
selected_competitions = competitions[
    competitions.apply(
        lambda row: (row["competition_name"], str(row["season_name"])) in SELECTED_COMPETITIONS
        and row.get("competition_gender") == "male"
        and not bool(row.get("competition_youth")),
        axis=1,
    )
].copy()

print("Selected competitions:")
display(selected_competitions[["competition_id", "season_id", "competition_name", "season_name"]])

match_records = []
for _, comp in selected_competitions.iterrows():
    match_path = DATA_DIR / "matches" / str(comp["competition_id"]) / f"{comp['season_id']}.json"
    if not match_path.exists():
        warnings.warn(f"Missing match file: {match_path}")
        continue
    for match in load_json(match_path):
        event_path = DATA_DIR / "events" / f"{match['match_id']}.json"
        match_records.append({
            "match_id": match["match_id"],
            "match_date": match.get("match_date"),
            "competition_name": comp["competition_name"],
            "season_name": str(comp["season_name"]),
            "home_team": match.get("home_team", {}).get("home_team_name"),
            "away_team": match.get("away_team", {}).get("away_team_name"),
            "home_score": match.get("home_score"),
            "away_score": match.get("away_score"),
            "event_file": str(event_path),
            "event_file_exists": event_path.exists(),
        })

matches_df = pd.DataFrame(match_records).sort_values(["match_date", "match_id"]).reset_index(drop=True)
print(f"Selected match rows: {matches_df.shape}")
display(matches_df.head())
display(matches_df.tail())
print("Event file availability:")
display(matches_df["event_file_exists"].value_counts().rename_axis("event_file_exists").reset_index(name="count"))

## 3. Build Leakage-Safe Shot Dataset

The target is `is_goal`, derived from `shot.outcome.name == "Goal"`. StatsBomb xG is retained for benchmark evaluation only and is not included in the model feature list.

In [ ]:
GOAL_X = 120.0
GOAL_Y = 40.0
GOAL_POST_Y_LOW = 36.0
GOAL_POST_Y_HIGH = 44.0

def safe_nested_name(obj, *keys):
    value = obj
    for key in keys:
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    if isinstance(value, dict):
        return value.get("name")
    return value

def shot_angle(x, y):
    # Angle between vectors from shot location to each goal post.
    v1 = np.array([GOAL_X - x, GOAL_POST_Y_LOW - y], dtype=float)
    v2 = np.array([GOAL_X - x, GOAL_POST_Y_HIGH - y], dtype=float)
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom <= 0:
        return np.pi
    cos_value = np.clip(np.dot(v1, v2) / denom, -1.0, 1.0)
    return float(np.arccos(cos_value))

def freeze_frame_features(event):
    shot_location = event.get("location") or [np.nan, np.nan]
    sx, sy = float(shot_location[0]), float(shot_location[1])
    frames = event.get("shot", {}).get("freeze_frame") or []
    teammates = 0
    opponents = 0
    opponents_in_box = 0
    opponents_goal_side = 0
    goalkeeper_distance = np.nan
    min_opponent_distance = np.nan
    opponent_distances = []
    for frame in frames:
        loc = frame.get("location") or [np.nan, np.nan]
        px, py = float(loc[0]), float(loc[1])
        distance = math.sqrt((px - sx) ** 2 + (py - sy) ** 2)
        if frame.get("teammate"):
            teammates += 1
        else:
            opponents += 1
            opponent_distances.append(distance)
            if px >= 102 and 18 <= py <= 62:
                opponents_in_box += 1
            if px >= sx:
                opponents_goal_side += 1
            position_name = safe_nested_name(frame, "position")
            if position_name == "Goalkeeper":
                goalkeeper_distance = distance if not np.isfinite(goalkeeper_distance) else min(goalkeeper_distance, distance)
    if opponent_distances:
        min_opponent_distance = float(min(opponent_distances))
    return {
        "freeze_teammate_count": teammates,
        "freeze_opponent_count": opponents,
        "freeze_opponents_in_box": opponents_in_box,
        "freeze_opponents_goal_side": opponents_goal_side,
        "freeze_goalkeeper_distance": goalkeeper_distance,
        "freeze_min_opponent_distance": min_opponent_distance,
        "freeze_frame_available": int(len(frames) > 0),
    }

def build_shot_rows(matches):
    rows = []
    excluded_period_5 = 0
    for _, match in matches[matches["event_file_exists"]].iterrows():
        events = load_json(match["event_file"])
        for event in events:
            if event.get("type", {}).get("name") != "Shot":
                continue
            period = event.get("period")
            if period in EXCLUDED_PERIODS:
                excluded_period_5 += 1
                continue
            shot = event.get("shot", {})
            loc = event.get("location") or [np.nan, np.nan]
            x = float(loc[0]) if len(loc) > 0 else np.nan
            y = float(loc[1]) if len(loc) > 1 else np.nan
            ff = freeze_frame_features(event)
            row = {
                "match_id": match["match_id"],
                "match_date": match["match_date"],
                "competition_name": match["competition_name"],
                "season_name": match["season_name"],
                "team": event.get("team", {}).get("name"),
                "opponent": match["away_team"] if event.get("team", {}).get("name") == match["home_team"] else match["home_team"],
                "period": period,
                "minute": event.get("minute", 0),
                "second": event.get("second", 0),
                "x": x,
                "y": y,
                "distance_to_goal": math.sqrt((GOAL_X - x) ** 2 + (GOAL_Y - y) ** 2) if np.isfinite(x) and np.isfinite(y) else np.nan,
                "angle_to_goal": shot_angle(x, y) if np.isfinite(x) and np.isfinite(y) else np.nan,
                "center_y_distance": abs(GOAL_Y - y) if np.isfinite(y) else np.nan,
                "shot_type": safe_nested_name(shot, "type"),
                "body_part": safe_nested_name(shot, "body_part"),
                "technique": safe_nested_name(shot, "technique"),
                "play_pattern": safe_nested_name(event, "play_pattern"),
                "position": safe_nested_name(event, "position"),
                "first_time": int(bool(shot.get("first_time"))),
                "aerial_won": int(bool(shot.get("aerial_won"))),
                "one_on_one": int(bool(shot.get("one_on_one"))),
                "open_goal": int(bool(shot.get("open_goal"))),
                "follows_dribble": int(bool(shot.get("follows_dribble"))),
                "statsbomb_xg": float(shot.get("statsbomb_xg")) if shot.get("statsbomb_xg") is not None else np.nan,
                "shot_outcome": safe_nested_name(shot, "outcome"),
                "is_goal": int(safe_nested_name(shot, "outcome") == "Goal"),
            }
            row.update(ff)
            rows.append(row)
    return pd.DataFrame(rows), excluded_period_5

shot_dataset, excluded_period_5_shots = build_shot_rows(matches_df)
shot_dataset.to_csv(OUTPUT_DIR / "custom_xg_shot_dataset.csv", index=False)

print(f"Shot rows after excluding period 5: {shot_dataset.shape}")
print(f"Excluded period 5 shootout shots: {excluded_period_5_shots}")
print(f"Goal rate: {shot_dataset['is_goal'].mean():.3f}")
display(shot_dataset.head())
display(shot_dataset[["shot_type", "body_part", "technique", "play_pattern", "shot_outcome", "statsbomb_xg", "is_goal"]].head())

assert 5 not in set(shot_dataset["period"].dropna().astype(int))
assert shot_dataset["statsbomb_xg"].notna().all()

## 4. Train/Test Split and Model Training

Shots are split by `match_id` using `GroupShuffleSplit`, which prevents shots from the same match appearing in both train and test sets.

In [ ]:
numeric_features = [
    "period", "minute", "second", "x", "y", "distance_to_goal", "angle_to_goal", "center_y_distance",
    "first_time", "aerial_won", "one_on_one", "open_goal", "follows_dribble",
    "freeze_teammate_count", "freeze_opponent_count", "freeze_opponents_in_box", "freeze_opponents_goal_side",
    "freeze_goalkeeper_distance", "freeze_min_opponent_distance", "freeze_frame_available",
]
categorical_features = ["shot_type", "body_part", "technique", "play_pattern", "position"]
feature_columns = numeric_features + categorical_features
leakage_columns = ["statsbomb_xg", "shot_outcome", "is_goal"]

for col in leakage_columns:
    assert col not in feature_columns, f"Leakage column included as feature: {col}"

model_df = shot_dataset.copy()
for col in categorical_features:
    model_df[col] = model_df[col].fillna("Unknown")
for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
    model_df[col] = model_df[col].fillna(model_df[col].median())

X = model_df[feature_columns]
y = model_df["is_goal"].astype(int)
groups = model_df["match_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_statsbomb_xg = model_df.iloc[test_idx]["statsbomb_xg"].clip(1e-6, 1 - 1e-6)

preprocess = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

custom_xg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        loss="log_loss",
        learning_rate=0.05,
        max_iter=250,
        max_leaf_nodes=20,
        l2_regularization=0.05,
        min_samples_leaf=25,
        random_state=RANDOM_STATE,
    )),
])

custom_xg_model.fit(X_train, y_train)
custom_test_pred = custom_xg_model.predict_proba(X_test)[:, 1]
custom_test_pred = np.clip(custom_test_pred, 1e-6, 1 - 1e-6)

print(f"Train shots: {len(X_train)} from {groups.iloc[train_idx].nunique()} matches")
print(f"Test shots: {len(X_test)} from {groups.iloc[test_idx].nunique()} matches")
print("Train/test match overlap:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0
assert np.all((custom_test_pred >= 0) & (custom_test_pred <= 1))

## 5. Model Evaluation vs StatsBomb xG

The custom model is evaluated against actual goals. StatsBomb xG is evaluated on the same test shots as a benchmark.

In [ ]:
def metric_row(model_name, probabilities):
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return {
        "model": model_name,
        "test_shots": len(y_test),
        "test_goals": int(y_test.sum()),
        "mean_prediction": float(probabilities.mean()),
        "actual_goal_rate": float(y_test.mean()),
        "log_loss": float(log_loss(y_test, probabilities, labels=[0, 1])),
        "brier_score": float(brier_score_loss(y_test, probabilities)),
        "roc_auc": float(roc_auc_score(y_test, probabilities)),
    }

metrics = pd.DataFrame([
    metric_row("Custom HistGradientBoosting xG", custom_test_pred),
    metric_row("StatsBomb provided xG", test_statsbomb_xg),
])
metrics.to_csv(OUTPUT_DIR / "custom_xg_model_metrics.csv", index=False)
display(metrics)

## 6. Calibration and Feature Importance

Calibration compares predicted probability buckets to actual goal rates. Feature importance uses permutation importance on the grouped test split.

In [ ]:
def calibration_table(name, probabilities, y_true, bins=np.linspace(0, 1, 11)):
    df = pd.DataFrame({"prediction": probabilities, "is_goal": np.asarray(y_true)})
    df["bucket"] = pd.cut(df["prediction"], bins=bins, include_lowest=True)
    table = df.groupby("bucket", observed=False).agg(
        shots=("is_goal", "size"),
        mean_prediction=("prediction", "mean"),
        actual_goal_rate=("is_goal", "mean"),
        goals=("is_goal", "sum"),
    ).reset_index()
    table["model"] = name
    table["bucket"] = table["bucket"].astype(str)
    return table

calibration = pd.concat([
    calibration_table("Custom HistGradientBoosting xG", custom_test_pred, y_test),
    calibration_table("StatsBomb provided xG", test_statsbomb_xg, y_test),
], ignore_index=True)
calibration.to_csv(OUTPUT_DIR / "custom_xg_calibration_table.csv", index=False)
display(calibration)

perm = permutation_importance(
    custom_xg_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_log_loss",
)
feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
feature_importance.to_csv(OUTPUT_DIR / "custom_xg_feature_importance.csv", index=False)
display(feature_importance.head(20))

## 7. Predict Custom xG for All Shots and Aggregate to Team-Match Level

This section generates custom xG for every selected non-shootout shot, then aggregates both custom xG and StatsBomb xG to team-match rows for the Spain-France pre-match comparison.

In [ ]:
all_custom_xg = custom_xg_model.predict_proba(model_df[feature_columns])[:, 1]
shot_dataset_with_predictions = shot_dataset.copy()
shot_dataset_with_predictions["custom_xg"] = np.clip(all_custom_xg, 1e-6, 1 - 1e-6)
shot_dataset_with_predictions.to_csv(OUTPUT_DIR / "custom_xg_shot_dataset_with_predictions.csv", index=False)

def aggregate_team_match_xg(shots, value_col):
    rows = []
    for _, match in matches_df[matches_df["event_file_exists"]].iterrows():
        home = match["home_team"]
        away = match["away_team"]
        match_shots = shots[
            (shots["match_id"] == match["match_id"])
            & (shots["period"].isin(HEADLINE_PERIODS))
        ]
        values = match_shots.groupby("team")[value_col].sum().to_dict()
        shot_counts = match_shots.groupby("team").size().to_dict()
        for team, opponent in [(home, away), (away, home)]:
            rows.append({
                "team": team,
                "opponent": opponent,
                "match_id": match["match_id"],
                "match_date": match["match_date"],
                "competition_name": match["competition_name"],
                "season_name": match["season_name"],
                "xg_for": float(values.get(team, 0.0)),
                "xg_against": float(values.get(opponent, 0.0)),
                "shots_for": int(shot_counts.get(team, 0)),
                "shots_against": int(shot_counts.get(opponent, 0)),
                "goals_for": match["home_score"] if team == home else match["away_score"],
                "goals_against": match["away_score"] if team == home else match["home_score"],
                "xg_source": value_col,
                "periods_included": ",".join(map(str, HEADLINE_PERIODS)),
            })
    return pd.DataFrame(rows).sort_values(["match_date", "match_id", "team"]).reset_index(drop=True)

custom_team_match = aggregate_team_match_xg(shot_dataset_with_predictions, "custom_xg")
statsbomb_team_match = aggregate_team_match_xg(shot_dataset_with_predictions, "statsbomb_xg")
custom_team_match.to_csv(OUTPUT_DIR / "custom_xg_team_match_dataset.csv", index=False)
statsbomb_team_match.to_csv(OUTPUT_DIR / "statsbomb_xg_team_match_dataset_for_comparison.csv", index=False)

display(custom_team_match[custom_team_match["team"].isin(["Spain", "France"])].tail(12))

## 8. Pre-Match Projection and Poisson Outcome Model

Both xG sources use the same downstream pre-match model: recency-weighted xG, shrinkage toward the global mean, neutral venue, and independent Poisson score probabilities.

In [ ]:
def weighted_average(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights)
    if valid.sum() == 0 or weights[valid].sum() <= 0:
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

def recent_profile(team_match_df, team):
    rows = team_match_df[team_match_df["team"] == team].copy()
    if rows.empty:
        raise RuntimeError(f"No team-match xG rows for {team}.")
    rows["match_date"] = pd.to_datetime(rows["match_date"])
    rows = rows.sort_values(["match_date", "match_id"]).tail(RECENT_MATCH_COUNT).reset_index(drop=True)
    rows["recency_weight"] = np.arange(1, len(rows) + 1)
    return {
        "team": team,
        "sample_matches": len(rows),
        "first_match_date": rows["match_date"].min().date().isoformat(),
        "last_match_date": rows["match_date"].max().date().isoformat(),
        "weighted_xg_for": weighted_average(rows["xg_for"], rows["recency_weight"]),
        "weighted_xg_against": weighted_average(rows["xg_against"], rows["recency_weight"]),
        "weighted_shots_for": weighted_average(rows["shots_for"], rows["recency_weight"]),
        "weighted_shots_against": weighted_average(rows["shots_against"], rows["recency_weight"]),
        "matches_used": "; ".join(f"{r.match_date.date().isoformat()} {r.team} vs {r.opponent}" for r in rows.itertuples()),
    }

def shrink_to_global(value, sample_matches, global_mean):
    if not np.isfinite(value):
        return global_mean
    return float((value * sample_matches + global_mean * SHRINKAGE_MATCHES) / (sample_matches + SHRINKAGE_MATCHES))

def poisson_pmf(k, lam):
    return math.exp(-lam) * lam**k / math.factorial(k)

def poisson_outcomes(spain_xg, france_xg):
    rows = []
    for spain_goals in range(MAX_GOALS + 1):
        for france_goals in range(MAX_GOALS + 1):
            prob = poisson_pmf(spain_goals, spain_xg) * poisson_pmf(france_goals, france_xg)
            rows.append({
                "spain_goals": spain_goals,
                "france_goals": france_goals,
                "scoreline": f"{spain_goals}-{france_goals}",
                "probability_raw": prob,
                "outcome": "Spain win" if spain_goals > france_goals else "France win" if france_goals > spain_goals else "Draw",
            })
    table = pd.DataFrame(rows)
    table["probability"] = table["probability_raw"] / table["probability_raw"].sum()
    table = table.sort_values("probability", ascending=False).reset_index(drop=True)
    outcome = table.groupby("outcome", as_index=False)["probability"].sum()
    outcome_map = dict(zip(outcome["outcome"], outcome["probability"]))
    return table, {
        "spain_win_90_probability": float(outcome_map.get("Spain win", 0.0)),
        "draw_90_probability": float(outcome_map.get("Draw", 0.0)),
        "france_win_90_probability": float(outcome_map.get("France win", 0.0)),
        "most_likely_scoreline": table.iloc[0]["scoreline"],
        "most_likely_scoreline_probability": float(table.iloc[0]["probability"]),
    }

def prematch_prediction(team_match_df, source_label):
    global_mean = float(team_match_df["xg_for"].mean())
    spain = recent_profile(team_match_df, "Spain")
    france = recent_profile(team_match_df, "France")
    for profile in [spain, france]:
        profile["global_average_xg"] = global_mean
        profile["shrunk_attack_xg"] = shrink_to_global(profile["weighted_xg_for"], profile["sample_matches"], global_mean)
        profile["shrunk_defense_xg_allowed"] = shrink_to_global(profile["weighted_xg_against"], profile["sample_matches"], global_mean)
    spain_projected = math.sqrt(max(spain["shrunk_attack_xg"], 0) * max(france["shrunk_defense_xg_allowed"], 0))
    france_projected = math.sqrt(max(france["shrunk_attack_xg"], 0) * max(spain["shrunk_defense_xg_allowed"], 0))
    scorelines, outcome = poisson_outcomes(spain_projected, france_projected)
    prediction = {
        "xg_source": source_label,
        "spain_projected_xg": spain_projected,
        "france_projected_xg": france_projected,
        **outcome,
        "predicted_90_minute_result": (
            "Spain win" if outcome["spain_win_90_probability"] > max(outcome["draw_90_probability"], outcome["france_win_90_probability"])
            else "France win" if outcome["france_win_90_probability"] > max(outcome["spain_win_90_probability"], outcome["draw_90_probability"])
            else "Draw"
        ),
        "global_average_xg": global_mean,
    }
    inputs = pd.DataFrame([spain, france])
    inputs["xg_source"] = source_label
    inputs["projected_match_xg"] = inputs["team"].map({"Spain": spain_projected, "France": france_projected})
    scorelines["xg_source"] = source_label
    return prediction, inputs, scorelines

custom_prediction, custom_inputs, custom_scorelines = prematch_prediction(custom_team_match, "Custom trained xG")
statsbomb_prediction, statsbomb_inputs, statsbomb_scorelines = prematch_prediction(statsbomb_team_match, "StatsBomb provided xG")

custom_prediction_df = pd.DataFrame([{**TARGET_MATCH, **custom_prediction}])
custom_prediction_df.to_csv(OUTPUT_DIR / "spain_france_custom_xg_prediction.csv", index=False)
custom_scorelines.to_csv(OUTPUT_DIR / "spain_france_custom_xg_scoreline_probabilities.csv", index=False)

comparison = pd.DataFrame([statsbomb_prediction, custom_prediction])
for col in ["spain_projected_xg", "france_projected_xg", "spain_win_90_probability", "draw_90_probability", "france_win_90_probability"]:
    statsbomb_value = comparison.loc[comparison["xg_source"] == "StatsBomb provided xG", col].iloc[0]
    comparison[f"{col}_difference_vs_statsbomb"] = comparison[col] - statsbomb_value
comparison.to_csv(OUTPUT_DIR / "spain_france_statsbomb_vs_custom_xg_comparison.csv", index=False)

inputs = pd.concat([statsbomb_inputs, custom_inputs], ignore_index=True)
inputs.to_csv(OUTPUT_DIR / "spain_france_custom_vs_statsbomb_xg_inputs.csv", index=False)

display(comparison)
display(custom_prediction_df)

for _, row in comparison.iterrows():
    assert np.isclose(row["spain_win_90_probability"] + row["draw_90_probability"] + row["france_win_90_probability"], 1.0)

## 9. Visual Comparison

These figures compare model evaluation, calibration, feature importance, projected xG, outcome probabilities, and the custom-model scoreline heatmap.

In [ ]:
# Calibration plot.
fig, ax = plt.subplots(figsize=(7, 5))
for model_name, group in calibration.dropna(subset=["mean_prediction", "actual_goal_rate"]).groupby("model"):
    ax.plot(group["mean_prediction"], group["actual_goal_rate"], marker="o", label=model_name)
ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set_title("Custom xG calibration vs StatsBomb benchmark")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Actual goal rate")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "custom_xg_calibration_plot.png", dpi=160, bbox_inches="tight")
plt.show()

# Feature importance.
top_features = feature_importance.head(15).sort_values("importance_mean")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_features["feature"], top_features["importance_mean"], xerr=top_features["importance_std"], color="#4c78a8")
ax.set_title("Custom xG permutation feature importance")
ax.set_xlabel("Importance, neg_log_loss decrease")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "custom_xg_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

# Projected xG comparison.
plot_xg = comparison.melt(
    id_vars="xg_source",
    value_vars=["spain_projected_xg", "france_projected_xg"],
    var_name="team_metric",
    value_name="projected_xg",
)
plot_xg["team"] = plot_xg["team_metric"].map({"spain_projected_xg": "Spain", "france_projected_xg": "France"})
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = np.arange(2)
sources = comparison["xg_source"].tolist()
for idx, source in enumerate(sources):
    values = [plot_xg[(plot_xg["xg_source"] == source) & (plot_xg["team"] == team)]["projected_xg"].iloc[0] for team in ["Spain", "France"]]
    ax.bar(x + (idx - 0.5) * width, values, width=width, label=source)
ax.set_xticks(x)
ax.set_xticklabels(["Spain", "France"])
ax.set_ylabel("Projected xG")
ax.set_title("StatsBomb xG vs custom trained xG projection")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "statsbomb_vs_custom_projected_xg.png", dpi=160, bbox_inches="tight")
plt.show()

# Outcome probability comparison.
outcome_plot = comparison.melt(
    id_vars="xg_source",
    value_vars=["spain_win_90_probability", "draw_90_probability", "france_win_90_probability"],
    var_name="outcome",
    value_name="probability",
)
outcome_plot["outcome"] = outcome_plot["outcome"].map({
    "spain_win_90_probability": "Spain win",
    "draw_90_probability": "Draw",
    "france_win_90_probability": "France win",
})
fig, ax = plt.subplots(figsize=(9, 5))
outcomes = ["Spain win", "Draw", "France win"]
x = np.arange(len(outcomes))
for idx, source in enumerate(sources):
    values = [outcome_plot[(outcome_plot["xg_source"] == source) & (outcome_plot["outcome"] == outcome)]["probability"].iloc[0] for outcome in outcomes]
    ax.bar(x + (idx - 0.5) * width, values, width=width, label=source)
ax.set_xticks(x)
ax.set_xticklabels(outcomes)
ax.set_ylim(0, 1)
ax.set_ylabel("Probability")
ax.set_title("StatsBomb xG vs custom xG outcome probabilities")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "statsbomb_vs_custom_outcome_probabilities.png", dpi=160, bbox_inches="tight")
plt.show()

# Custom xG scoreline heatmap.
heatmap = custom_scorelines.pivot(index="france_goals", columns="spain_goals", values="probability").sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(heatmap.values, cmap="magma", aspect="auto")
ax.set_title("Custom xG scoreline probability heatmap")
ax.set_xlabel("Spain goals")
ax.set_ylabel("France goals")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
for y_idx in range(heatmap.shape[0]):
    for x_idx in range(heatmap.shape[1]):
        value = heatmap.values[y_idx, x_idx]
        if value >= 0.015:
            ax.text(x_idx, y_idx, f"{value:.1%}", ha="center", va="center", color="white", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Probability")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "custom_xg_scoreline_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Comparison Report

The report explains what the custom model learned, how it differs from StatsBomb xG, and how the final Spain-France prediction changes.

In [ ]:
def pct(value):
    return f"{float(value):.1%}"

stats_row = comparison[comparison["xg_source"] == "StatsBomb provided xG"].iloc[0]
custom_row = comparison[comparison["xg_source"] == "Custom trained xG"].iloc[0]
custom_metric = metrics[metrics["model"] == "Custom HistGradientBoosting xG"].iloc[0]
sb_metric = metrics[metrics["model"] == "StatsBomb provided xG"].iloc[0]

report = f"""
# Custom xG vs StatsBomb xG: Spain-France Comparison

## Data and model

This notebook trained a custom supervised xG model from local StatsBomb shot events in FIFA World Cup 2018/2022 and UEFA Euro 2020/2024. Period 5 shootout shots were excluded. The model used pre-shot features only: location, distance, angle, period, time, shot type, body part, technique, play pattern, boolean shot context flags, and freeze-frame counts.

StatsBomb's `shot.statsbomb_xg` was not used as a training feature. It was kept only as a benchmark.

## Test-set evaluation

- Custom model log loss: **{custom_metric['log_loss']:.3f}**
- StatsBomb xG log loss on the same test shots: **{sb_metric['log_loss']:.3f}**
- Custom model Brier score: **{custom_metric['brier_score']:.3f}**
- StatsBomb xG Brier score: **{sb_metric['brier_score']:.3f}**
- Custom model ROC AUC: **{custom_metric['roc_auc']:.3f}**
- StatsBomb xG ROC AUC: **{sb_metric['roc_auc']:.3f}**

## Spain-France projected xG

- StatsBomb xG projection: Spain **{stats_row['spain_projected_xg']:.2f}**, France **{stats_row['france_projected_xg']:.2f}**
- Custom xG projection: Spain **{custom_row['spain_projected_xg']:.2f}**, France **{custom_row['france_projected_xg']:.2f}**

## 90-minute outcome probabilities

StatsBomb xG workflow:

- Spain win: **{pct(stats_row['spain_win_90_probability'])}**
- Draw: **{pct(stats_row['draw_90_probability'])}**
- France win: **{pct(stats_row['france_win_90_probability'])}**
- Most likely scoreline: **{stats_row['most_likely_scoreline']}**

Custom xG workflow:

- Spain win: **{pct(custom_row['spain_win_90_probability'])}**
- Draw: **{pct(custom_row['draw_90_probability'])}**
- France win: **{pct(custom_row['france_win_90_probability'])}**
- Most likely scoreline: **{custom_row['most_likely_scoreline']}**

## Interpretation

The custom model lets us test whether a locally trained xG estimate changes the final match view. Differences should be interpreted cautiously because this training set is much smaller than a professional xG model's data and uses a compact feature set. The comparison is useful as a learning experiment, not as a betting-grade forecast.
""".strip()

report_path = OUTPUT_DIR / "spain_france_custom_xg_report.md"
report_path.write_text(report, encoding="utf-8")
display(Markdown(report))
print(f"Saved {report_path}")

## 11. Final Checklist

This cell verifies leakage controls, grouped splitting, probability bounds, probability sums, and all requested outputs.

In [ ]:
required_outputs = [
    OUTPUT_DIR / "custom_xg_shot_dataset.csv",
    OUTPUT_DIR / "custom_xg_model_metrics.csv",
    OUTPUT_DIR / "custom_xg_calibration_table.csv",
    OUTPUT_DIR / "custom_xg_feature_importance.csv",
    OUTPUT_DIR / "custom_xg_team_match_dataset.csv",
    OUTPUT_DIR / "spain_france_custom_xg_prediction.csv",
    OUTPUT_DIR / "spain_france_statsbomb_vs_custom_xg_comparison.csv",
    OUTPUT_DIR / "spain_france_custom_xg_scoreline_probabilities.csv",
    OUTPUT_DIR / "spain_france_custom_xg_report.md",
    OUTPUT_DIR / "custom_xg_calibration_plot.png",
    OUTPUT_DIR / "custom_xg_feature_importance.png",
    OUTPUT_DIR / "statsbomb_vs_custom_projected_xg.png",
    OUTPUT_DIR / "statsbomb_vs_custom_outcome_probabilities.png",
    OUTPUT_DIR / "custom_xg_scoreline_heatmap.png",
]

checks = [
    ("No period 5 shots used for training", 5 not in set(shot_dataset["period"].dropna().astype(int))),
    ("StatsBomb xG excluded from model features", "statsbomb_xg" not in feature_columns),
    ("Outcome excluded from model features", "shot_outcome" not in feature_columns and "is_goal" not in feature_columns),
    ("Grouped train/test split by match_id", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0),
    ("Custom xG probabilities within [0, 1]", shot_dataset_with_predictions["custom_xg"].between(0, 1).all()),
    ("Custom Poisson probabilities sum to 1", np.isclose(custom_row["spain_win_90_probability"] + custom_row["draw_90_probability"] + custom_row["france_win_90_probability"], 1.0)),
    ("StatsBomb Poisson probabilities sum to 1", np.isclose(stats_row["spain_win_90_probability"] + stats_row["draw_90_probability"] + stats_row["france_win_90_probability"], 1.0)),
    ("All requested outputs generated", all(path.exists() for path in required_outputs)),
]

for label, ok in checks:
    print(f"[{'Completed' if ok else 'Failed'}] {label}")

print("\nHeadline comparison:")
display(comparison)

print("\nFinal output file paths:")
for path in required_outputs:
    print(f"- {path} [{'exists' if path.exists() else 'missing'}]")